In [18]:
import re
from collections import defaultdict

import pandas as pd


In [19]:
# === Словарь паттернов ===
COUNTRY_KEYWORDS = {
    "Австралия": [
        r"\bавстрали", r"\bканберр", r"\bсидне", r"\bмельбурн",
    ],
    "Австрия": [
        r"\bавстри", r"\bвена", r"\bзальцбург",
    ],
    "Азербайджан": [
        r"\bазербайджан", r"\bбаку", r"\bнагорн[а-я\-]*\s*карабах[а-я]*\b",
    ],
    "Албания": [
        r"\bалбан", r"\bтиран",
    ],
    "Алжир": [
        r"\bалжир",
    ],
    "Аргентина": [
        r"\bаргентин", r"\bбуэнос",
    ],
    "Армения": [
        r"\bармени", r"\bереван",
    ],
    "Афганистан": [
        r"\bафган", r"\bкабул", r"\bталиб",
    ],
    "Бангладеш": [
        r"\bбангладеш", r"\bдакка",
    ],
    'Беларусь': [
        r'\bбеларус', r'\bбелорус', r'\bлукашенк'
    ],
    "Бельгия": [
        r"\bбельги", r"\bбрюссел",
    ],
    "Болгария": [
        r"\bболгар", r"\bсофи",
    ],
    "Боливия": [
        r"\bболиви",
    ],
    "Босния и Герцеговина": [
        r"\bбосни[а-я]*\s*(и)?\s*герцеговин[а-я]*"
    ],
    "Бразилия": [
        r"\bбразил", r"\bбразилиа", r"\bсан-паул", r"\bсалвадор", r'\bрио-де-жанейро'
    ],
    "Ватикан": [
        r"\bватикан", r"\bпапа римск\b",
    ],
    'Великобритания': [
        r'\bбритан', r'\bвеликобритан', r'\bлондон', r'\bангли',
    ],
    "Венгрия": [
        r"\bвенгр", r'\bбудапешт',
    ],
    "Венесуэла": [
        r"\bвенесуэл",
    ],
    "Гана": [
        r"\bгана", r"\bаккр",
    ],
    "Гватемала": [
        r"\bгватемал",
    ],
    'Германия': [
        r'\bгерман', r'\bберлин', r'\bкельн',
    ],
    "Греция": [
        r"\bгреци", r"\bафин",
    ],
    "Грузия": [
        r"\bгрузи", r"\bтбилис",
    ],
    "Дания": [
        r"\bдани", r"\bкопенгаген", r'\bдатчане',
    ],
    "Зимбабве": [
        r"\bзимбабв",
    ],
    'Израиль': [
        r'\bизраил', r'\bтель-авив', r'\bиерусалим',
    ],
    "Индонезия": [
        r"\bиндонез", r"\bджакарт",
    ],
    "Ирак": [
        r"\bирак", r"\bбагдад",
    ],
    "Ирландия": [
        r"\bирланд", r"\bдублин",
    ],
    "Испания": [
        r"\bиспани", r"\bмадрид", r"\bбарселон",
    ],
    'Индия': [
        r"\bинд(?:ия|ийск)[а-я]*", r"\bдели", r"\bмумба[и-я]*", r"\bиндии\b"
    ],
    'Иран': [
        r'\bиран', r'\bтегеран',
    ],
    "Италия": [
        r"\bитал[а-я]*", r"\bрим", r"\bмилан", r"\bнеапол", r"\bфлоренц", r"\bтурин",
        r"\bвенеци"
    ],
    "Йемен": [
        r"\bйемен", r'\bхусит',
    ],
    "Камбоджа": [
        r"\bкамбодж", r"\bпномпен",
    ],
    "Камерун": [
        r"\bкамерун", r"\bяунд",
    ],
    "Канада": [
        r"\bканад", r"\bоттав", r"\bторонт", r"\bмонреал", r"\bванкувер",
    ],
    "Казахстан": [
        r"\bказахстан", r"\bастан", r"\bалмат", r"\bалма-ат", r"\bкараганд"
    ],
    "Кипр": [
        r"\bкипр[а-я]*", r'\bникос',
    ],
    "Киргизия": [
        r"\bкыргыз", r"\bкиргиз", r"\bбишкек",
    ],
    'Китай': [
        r'\bкита', r'\bпекин', r'си цзиньпин',
    ],
    "Коморы": [
        r"\bкомор", r"\bморони"
    ],
    "Косово": [
        r"\bкосов", r"\bприштин"
    ],
    "Кот-д’Ивуар": [
        r"\bабиджан", r"\bкот(?:[-–—\s]?д['’]?)\s*ивуар[а-я]*",
    ],
    "Латвия": [
        r"\bлатви", r"\bриг[а-я]\b",
    ],
    "Ливан": [
        r"\bливан", r"\bбейрут"
    ],
    "Литва": [
        r"\bлитв", r"\bвильнюс",
    ],
    "Лихтенштейн": [
        r"\bлихтенштейн",
    ],
    "Македония": [
        r"\bмакедон",
    ],
    "Мальдивы": [
        r"\bмальдив",
    ],
    "Мексика": [
        r"\bмексик", r"\bмехико"
    ],
    "Мозамбик": [
        r"\bмозамбик", r"\bмапут",
    ],
    "Молдова": [
        r"\bмолдов", r"\bмолдав", r'\bкишин[её]в'
    ],
    "Нигерия": [
        r"\bнигери", r"\bабудж",
    ],
    "Нидерланды": [
        r"\bнидерланд", r"\bголланд", r"\bамстердам", r"\bроттердам", r"\bгааг",
    ],
    "Пакистан": [
        r"\bпакистан", r"\bисламабад", r"\bкарач", r"\bлахо", r"\bпенджаб"
    ],
    'Палестина': [
        r'\bхамас', r"\bсектор[а-я]*\s+газ[а-я]*\b"
    ],
    "Перу": [
        r"\bперу", r"\bлима",
    ],
    "Польша": [
        r"\bпольш", r"\bваршав",
    ],
    "Португалия": [
        r"\bпортугал", r"\bлиссабон",
    ],
    'Россия': [
        r'\bросси', r'\bрусск', r"\bроссийск.*федерац", r'\bрпц', r'\bмоскв', r'\bмосков', r'\bрф', r'\bчечен',
        r'\bчечн',
        r'\bсанкт[-\s]?петербур', r'\bпутин', r'\bгазпром', r'\btelegram\b', r'\bссср\b', r'\bхабаровск', r'\bомск',
        r'\bволгоград', r'\bкалининград', r'\bвладикавказ', r'байкал',
        r'\bастрахан', r'\bдагестан', r'\bсахалин', r'\bкурск\b',
        r'\bшереметь'
    ],
    "Румыния": [
        r"\bрумын", r"\bбухарест",
    ],
    "Сальвадор": [
        r"\bсальвадор",
    ],
    "Сан-Марино": [
        r"\bсан-марин", r"\bсерравалл"
    ],
    "Северная Корея": [
        r"\bсеверн.*коре", r"\bпхеньян", r'\bкндр\b'
    ],
    "Сенегал": [
        r"\bсенегал", r"\bдакар",
    ],
    "Сербия": [
        r"\bсерб", r"\bбелград",
    ],
    "Сингапур": [
        r"\bсингапур", r'\bsingapore\b'
    ],
    "Сирия": [
        r"\bсири", r"\bдамаск", r"\bалепп",
    ],
    "Словакия": [
        r"\bсловаки", r"\bбратислав",
    ],
    "Словения": [
        r"\bсловен",
    ],
    "Суринам": [
        r"\bсуринам",
    ],
    'США': [
        r'\bсша\b', r'\bамерик', r'\bвашингтон', r'\bбайден', r'\bбуш', r'\bобам\b', r'\bmicrosoft\b',
        r'\baol\b', r'\bНью-Йорк'
        r'\btime warner\b', r'\bapple\b', r'\bgoogle\b', r'\bfacebook\b', r'\byoutube\b', r'\bsony\b', r'\bоон\b'
    ],
    'Таджикистан': [
        r'\bтаджик', r'\bдушанабе'
    ],
    "Таиланд": [
        r"\bтаиланд", r"\bбангкок",
    ],
    "Тайвань": [
        r"\bтайван", r'\bтайбэй'
    ],
    "Туркменистан": [
        r"\bтуркмен", r"\bашхабад",
    ],
    'Турция': [
        r'\bтурци', r'\bэрдоган', r'\bанкар',
    ],
    "Уганда": [
        r"\bуганд",
    ],
    'Узбекистан': [
        r'\bузбек',
    ],
    'Украина': [
        r'украин', r'\bкиев', r'\bзеленск', r'\bдонецк', r'\bлуганск',
    ],
    "Уругвай": [
        r"\bуругва", r'\bмонтевидео',
    ],
    "Филиппины": [
        r'\bфилиппин'
    ],
    "Финляндия": [
        r"\bфинлян", r"\bхельсинк",
    ],
    'Франция': [
        r'\bфранц', r'\bпариж', r'\bмакрон',
    ],
    'Хорватия': [
        r'\bхорват'
    ],
    "Черногория": [
        r"\bчерногор", r"\bподгориц",
    ],
    "Чехия": [
        r"\bчехи", r"\bпраг",
    ],
    "Чили": [
        r"\bчили", r"\bсантьяг"
    ],
    "Шотландия": [
        r"\bшотланд", r"\bединбург", r"\bглазг",
    ],
    'Щвейцария': [
        r'\bшвейцар', r'\bберн'
    ],
    "Швеция": [
        r"\bшвед", r"\bстокгольм",
    ],
    "Эквадор": [
        r"\bэквадор", r"\bкито",
    ],
    "Эритрея": [
        r"\bэритре[яи]", r"\bасмэр", r"\bмассауа"
    ],
    "Эстония": [
        r"\bэстони", r"\bталлин",
    ],
    "Эфиопия": [
        r"\bэфиоп", r"\bаддис-абеб"
    ],
    "Югославия": [
        r"\bюгослав"
    ],
    "ЮАР": [
        r"\bюар\b", r"\bюжн.*африк"
    ],
    "Южная Корея": [
        r"\bюжн.*коре", r"\bсеул", r"\bпусан",
    ],
    "Япония": [
        r"\bяпон", r"\bтокио", r"\bтокий", r"\bхиросим", r"\bосака",
    ],
    'UNIDENTIFIED': []
}


def detect_countries(text: str, keywords: dict = COUNTRY_KEYWORDS, threshold: int = 1):
    """
    Определяет страну (или несколько) по тексту события.
    threshold — минимальное количество совпадений, чтобы считать страну найденной.
    Возвращает: список стран, отсортированный по убыванию количества совпадений.
    """
    text = lemmatize(morph, text)
    counts = defaultdict(int)

    for country, patterns in keywords.items():
        for pattern in patterns:
            matches = re.findall(pattern, text)
            if matches:
                counts[country] += len(matches)

    if not counts:
        return ['UNIDENTIFIED']

    # Оставляем страны с количеством совпадений >= threshold
    filtered = {c: n for c, n in counts.items() if n >= threshold}
    if not filtered:
        return ['UNIDENTIFIED']

    # Сортируем по числу совпадений
    ranked = sorted(filtered.items(), key=lambda x: x[1], reverse=True)
    return [c for c, _ in ranked]


def classify_events(df: pd.DataFrame, text_column: str = 'event'):
    """
    Добавляет колонку 'countries' на основе текстов событий.
    """
    df = df.copy()
    df['countries'] = df[text_column].apply(lambda text: detect_countries(text))
    return df


import pymorphy3

morph = pymorphy3.MorphAnalyzer()


def lemmatize(morph, text):
    # токенизация (удаляем всё, кроме букв)
    tokens = re.findall(r"[а-яА-ЯёЁ]+", text.lower())

    # лемматизация
    lemmas = [morph.parse(word)[0].normal_form for word in tokens]
    return " ".join(lemmas)

In [20]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df = classify_events(df)
undefined_mask = df['countries'].apply(lambda x: 'UNIDENTIFIED' in x)
undefined_count = undefined_mask.sum()

print(f'Всего строк: {len(df)}. Кол-во неопознанных строк: {undefined_count}')

# Вернуть только неопознанные строки
unidentified_df = df[undefined_mask].copy()
# df.head()  # 1543 неопознанных из 5661 событий
df

Всего строк: 5644. Кол-во неопознанных строк: 1576


,date_start,date_end,event,countries
0,2000-01-01,NaN,Деноминация белорусского рубля;,[Беларусь]
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н...",[Иран]
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог...",[Великобритания]
3,2000-01-02,NaN,Крушение украинского сухогруза типа «река-море...,"[Камбоджа, Украина]"
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...,"[Ливан, Россия]"
...,...,...,...,...
5639,2025-09-18,NaN,На камчатке зафиксировано землетрясение магнит...,[UNIDENTIFIED]
5640,2025-09-20,NaN,Проведение конкурса песни «интервидение» в мос...,[Россия]
5641,2025-09-23,NaN,Международный уголовный суд представил подтвер...,[Филиппины]
5642,2025-09-25,NaN,Парламент Кыргызстана объявил о самороспуске.,[Киргизия]


In [21]:
unidentified_df

,date_start,date_end,event,countries
12,2000-01-10,NaN,Объявление компаний «AOL» и «Time Warner» о пл...,[UNIDENTIFIED]
25,2000-01-24,NaN,В ночь с 25 на 26 января — крушение поезда на ...,[UNIDENTIFIED]
44,2000-02-16,2000-02-20,проведение зимних Игр Доброй воли в Лейк-Плэсиде.,[UNIDENTIFIED]
45,2000-02-17,NaN,Выход «microsoft windows 2000».,[UNIDENTIFIED]
60,2000-03-04,NaN,Sony выпустила игровую приставку 6-го поколени...,[UNIDENTIFIED]
...,...,...,...,...
5630,2025-09-09,NaN,Антиправительственные акции протеста в Непале....,[UNIDENTIFIED]
5632,2025-09-10,NaN,Убийство политического активиста чарли кирка в...,[UNIDENTIFIED]
5633,2025-09-11,2025-09-14,турнир по Dota 2 The International 2025.,[UNIDENTIFIED]
5638,2025-09-16,NaN,Всеобщие выборы в малави.,[UNIDENTIFIED]
